# How to Fine-Tune a Decision Model with Unsloth (SFT + LoRA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/finetuning/notebooks/sft_for_decision_models_with_unsloth.ipynb)

This tutorial shows you how to fine-tune a decision model, a model that makes decisions instead of generating text, on your own labels.

We fine-tune our [`d1-3B`](https://huggingface.co/LiquidAI/d1-3B) decision model with Supervised Fine-tuning (SFT) and LoRA using Unsloth, with intent classification for customer support routing on Banking77 as the example.

Follow along if it's your first time using Unsloth, or take single code snippets for your own workflow

## 🎯 What You'll Find:
- **LoRA + SFT** - Fine-tune the d1-3B decision model on your own decisions using LoRA (from PEFT)
- **Evaluation** - Compare the decision model's accuracy before and after fine-tuning

## 📋 Prerequisites:
- **GPU Runtime**: Select a GPU in `Runtime` → `Change runtime type`. A free T4 GPU is enough (about 30 minutes of training)
- **Hugging Face Account**: Only needed to push your model to the Hub

### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==5.17.0 "torchao>0.16.0"
!pip install --no-deps trl==1.15.0

### Loading d1-3B

d1-3B ships its own code for the System One API (`system_one` and `system_one_batch`), so we load it with `trust_remote_code = True`.

We also download the model repository locally. It contains `prompt.py`, the module d1-3B uses to turn a state and a question into a prompt. We will reuse it to build our training data, so that the model sees the exact same prompt during training and inference.

Unsloth needs to be imported before Transformers, so we import it first.

In [2]:
from unsloth import FastVisionModel # Import Unsloth before Transformers!
import torch
import sys
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer

model_id = "LiquidAI/d1-3B"

# bfloat16 on GPUs that support it natively (e.g. A100), float16 otherwise (e.g. T4)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation = False) else torch.float16

# Download the checkpoint and d1-3B's own code, and make its prompt builder importable
local_dir = snapshot_download(model_id)
sys.path.insert(0, local_dir)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(
    model_id,
    trust_remote_code = True,
    dtype = dtype,
).to("cuda")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

A new version of the following files was downloaded from https://huggingface.co/LiquidAI/d1-3B:
- prompt.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/LiquidAI/d1-3B:
- hybrid.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/LiquidAI/d1-3B:
- api.py
- lfm2_vl.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/LiquidAI/d1-3B:
- runner.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions 

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

### What is a decision model?

A decision model is a model that makes decisions instead of generating text: it answers yes/no, multiple-choice, and scoring questions with a probability for each answer, in a single forward pass. Typical uses are classification, routing and triage, content moderation, and LLM-as-a-judge scoring.

d1-3B is a 3B parameter decision model built on [LFM2.5-VL-3B](https://huggingface.co/LiquidAI/LFM2.5-VL-3B). It takes a state and a set of questions, and returns typed answers in one forward pass, without generating any output tokens. Each question uses one of the following three primitives:

- `noul`: Yes or no, as P(yes)
- `choice`: One of named options, with a probability for each
- `score`: A level on an ordered scale of 2 to 10 levels

Before we fine-tune, let's see how d1-3B makes a decision. We ask a `choice` question about a customer message:

In [3]:
state = "I was charged twice this month, please refund one of them."

questions = {
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this?",
        "criteria": {
            "billing": "Charges, refunds, invoices",
            "technical": "App or site faults",
            "fraud": "Suspected unauthorised use",
        },
    },
}

response = model.system_one(state = state, questions = questions)

print(response)

{'answers': {'team': {'type': 'choice', 'choice': 'billing', 'confidence': 0.9835613206479191, 'probabilities': {'billing': 0.9835613206479191, 'technical': 0.01059020995608956, 'fraud': 0.005848469395991271}}}, 'usage': {'input_tokens': 61, 'output_tokens': 0}}


Under the hood, d1-3B renders the state and the question into a single prompt. Each option gets a short code (`A`, `B`, `C`, ...), and the prompt ends right after the assistant header:

In [4]:
from prompt import aliases, as_question, render

rendered_prompt = render(
    tokenizer = tokenizer,
    state = state,
    q = as_question(questions["team"]),
    bos = tokenizer.bos_token,
)

print(rendered_prompt)

<|startoftext|><|im_start|>user
I was charged twice this month, please refund one of them.


QUESTION:
Which team should handle this?

Options:
A Charges, refunds, invoices
B App or site faults
C Suspected unauthorised use

Reply with the option code only.<|im_end|>
<|im_start|>assistant



The model then runs one forward pass and reads the next-token probabilities at the end of the prompt, the **answer slot**. It only looks at the tokens of the option codes and normalizes them into the probabilities you saw above.

This means that fine-tuning a decision model is regular SFT where **the completion is a single token: the code of the correct option**, followed by the end-of-turn token `<|im_end|>`.

Let's free up the GPU memory before we load the model for training:

In [5]:
import gc

del model
gc.collect()
torch.cuda.empty_cache()

### Unsloth

For training, we load the same weights with Unsloth's `FastVisionModel`. d1-3B's checkpoint uses the standard LFM2-VL module names, so we don't need its custom code here.

**Why not `FastDecisionModel`?** Unsloth also has `FastDecisionModel` and `DecisionTrainer` for training decision models. Given d1-3B, they would add a new, randomly initialized decision head on top of the model and save the result in Unsloth's own decision-model format. You would lose d1-3B's trained readout and its System One API. Instead, we fine-tune d1-3B's own readout: the model keeps answering at the answer slot, and the fine-tuned model works with `system_one` exactly like d1-3B.

In [6]:
from unsloth import FastVisionModel
from transformers import AutoModelForImageTextToText

model, processor = FastVisionModel.from_pretrained(
    model_name = model_id,
    # d1-3B's config only registers its own AutoModel class, so we name the standard class with the language model head
    auto_model = AutoModelForImageTextToText,
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = False, # 4 bit quantization to reduce memory
)

==((====))==  Unsloth 2026.10.5: Fast Lfm2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

We now add LoRA adapters so we only need to update a small amount of parameters!

Our task is text-only, so we only adapt the language model: its attention layers (`q_proj`, `k_proj`, `v_proj`, `out_proj`), its short convolution layers (`in_proj`, `out_proj`), and its MLP layers (`w1`, `w2`, `w3`). Unsloth's `finetune_*_layers` options select layers by name and skip LFM2's convolution layers, so we pass a regex as `target_modules` instead.

In [7]:
model = FastVisionModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    # Only the language model's layers, not the vision encoder's
    target_modules = r".*language_model.*\.(q_proj|k_proj|v_proj|out_proj|in_proj|w1|w2|w3)",
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

<a name="Data"></a>
### Data Prep
We will use [Banking77](https://huggingface.co/datasets/mteb/banking77), a dataset of ~13k online banking customer messages, each labeled with one of 77 fine-grained intents. We train on 3,000 messages and evaluate on 100 to keep things fast. Feel free to increase these numbers with `select(range(...))`.

In [8]:
from datasets import load_dataset
dataset = load_dataset("mteb/banking77")
train_dataset = dataset["train"].shuffle(seed = 42).select(range(3000))
test_dataset = dataset["test"].shuffle(seed = 42).select(range(100))
dataset

README.md:   0%|          | 0.00/16.6k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  294kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 91.4kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9993 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3076 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 9993
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 3076
    })
})

Let's see how the first row looks like!

In [9]:
train_dataset[0]

{'text': 'How long is a top up supposed to be pending, because an hour is too long! Can you fix this to go through?',
 'label': 47,
 'label_text': 'pending_top_up'}

We turn the classification task into a single `choice` question. Each intent becomes one option, described by its label name.

**Important**: The option codes are assigned by position. Use the same question, with the options in the same order, for training and inference.

In [10]:
labels = sorted(set(dataset["train"]["label_text"]))

intent_question = {
    "type": "choice",
    "instructions": "Which intent does this banking customer message express?",
    "criteria": {label: label.replace("_", " ") for label in labels},
}

# The code d1-3B assigns to each option, e.g. "card_arrival" -> "12"
codes = {label: code for label, (code, _) in zip(labels, aliases(tokenizer, labels))}

print(f"Number of options: {len(labels)}")
print(f"First option codes: {dict(list(codes.items())[:5])}")

Number of options: 77
First option codes: {'Refund_not_showing_up': '00', 'activate_my_card': '01', 'age_limit': '02', 'apple_pay_or_google_pay': '03', 'atm_support': '04'}


With up to 26 options, d1-3B uses the letters `A` to `Z` as codes. With more options, as here, it switches to the two-digit codes `00` to `76`, each of which is a single token.

We now render every message into d1-3B's prompt, append the code of the correct intent followed by `<|im_end|>`, and save it to `text`. We also hold out 10% of the training data for evaluation during training.

In [11]:
question = as_question(intent_question)

def formatting_prompts_func(example):
    rendered_prompt = render(
        tokenizer = tokenizer,
        state = example["text"],
        q = question,
        bos = tokenizer.bos_token,
    )
    return { "text" : rendered_prompt + codes[example["label_text"]] + tokenizer.eos_token }

split = train_dataset.train_test_split(test_size = 0.1, seed = 42)
train_dataset = split["train"].map(formatting_prompts_func, remove_columns = split["train"].column_names)
eval_dataset = split["test"].map(formatting_prompts_func, remove_columns = split["test"].column_names)

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [12]:
print(train_dataset[0]["text"])

<|startoftext|><|im_start|>user
So, I was under the impression that you have an offer for crypto top up. This doesn't seem to be working. Now, the money has just disappeared from my account. What is the deal?


QUESTION:
Which intent does this banking customer message express?

Options:
00 Refund not showing up
01 activate my card
02 age limit
03 apple pay or google pay
04 atm support
05 automatic top up
06 balance not updated after bank transfer
07 balance not updated after cheque or cash deposit
08 beneficiary not allowed
09 cancel transfer
10 card about to expire
11 card acceptance
12 card arrival
13 card delivery estimate
14 card linking
15 card not working
16 card payment fee charged
17 card payment not recognised
18 card payment wrong exchange rate
19 card swallowed
20 cash withdrawal charge
21 cash withdrawal not recognised
22 change pin
23 compromised card
24 contactless not working
25 country support
26 declined card payment
27 declined cash withdrawal
28 declined transfer
29 

<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We train for one full epoch. Our data is text-only, so we pass the text tokenizer instead of the processor.

In [13]:
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model) # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 0.05, # A float is a fraction of the total steps
        num_train_epochs = 1, # Set this for 1 full training run.
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = 50,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2700 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/300 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. For a decision model, the assistant output is the option code.

In [14]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the first row again.

In [15]:
print(tokenizer.decode(trainer.train_dataset[0]["input_ids"]))

<|startoftext|><|im_start|>user
So, I was under the impression that you have an offer for crypto top up. This doesn't seem to be working. Now, the money has just disappeared from my account. What is the deal?


QUESTION:
Which intent does this banking customer message express?

Options:
00 Refund not showing up
01 activate my card
02 age limit
03 apple pay or google pay
04 atm support
05 automatic top up
06 balance not updated after bank transfer
07 balance not updated after cheque or cash deposit
08 beneficiary not allowed
09 cancel transfer
10 card about to expire
11 card acceptance
12 card arrival
13 card delivery estimate
14 card linking
15 card not working
16 card payment fee charged
17 card payment not recognised
18 card payment wrong exchange rate
19 card swallowed
20 cash withdrawal charge
21 cash withdrawal not recognised
22 change pin
23 compromised card
24 contactless not working
25 country support
26 declined card payment
27 declined cash withdrawal
28 declined transfer
29 

Now let's print the masked out example - you should see only the option code and `<|im_end|>`:

In [16]:
print(tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ").strip())

61<|im_end|>


In [17]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
5.934 GB of memory reserved.


In [18]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,700 | Num Epochs = 1 | Total steps = 169
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 24,461,312 of 3,147,945,200 (0.78% trained)


Unsloth: Will smartly offload gradients to save VRAM!


`causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,0.325225,0.305796
100,0.183305,0.223051
150,0.247102,0.205309
169,0.160880,0.203071


Filter:   0%|          | 0/300 [00:00<?, ? examples/s]

Unsloth: Not an error, but `num_items_in_batch` could not be counted for Lfm2VlForConditionalGeneration's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [19]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

1653.8011 seconds used for training.
27.56 minutes used for training.
Peak reserved memory = 7.115 GB.
Peak reserved memory for training = 1.181 GB.
Peak reserved memory % of max memory = 48.857 %.
Peak reserved memory for training % of max memory = 8.11 %.


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit, scroll down!

In [20]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/chat_template.jinja',
 'lora_model/tokenizer.json')

### Saving to float16

To use the System One API with your fine-tuned model, merge the LoRA adapters into the model and save it in `float16`. The merged checkpoint contains only the weights and the tokenizer, so we also copy d1-3B's own code and processor configuration into the same folder.

In [21]:
import shutil

output_dir = "d1-3B-banking77-merged"

# Merge to 16bit
model.save_pretrained_merged(output_dir, tokenizer, save_method = "merged_16bit",)

# d1-3B's code for the System One API, its processor configuration, and its license
for filename in ["api.py", "hybrid.py", "lfm2_vl.py", "modeling_d1.py", "prompt.py", "runner.py", "processor_config.json", "LICENSE"]:
    shutil.copy(os.path.join(local_dir, filename), output_dir)

# To push the merged model and d1-3B's code to the Hugging Face Hub, uncomment:
# from huggingface_hub import create_repo, upload_folder
# create_repo("your-model-path", exist_ok = True)
# upload_folder(folder_path = output_dir, repo_id = "your-model-path")

Unsloth: Restored added_tokens_decoder metadata in d1-3B-banking77-merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `d1-3B-banking77-merged`: 100%|██████████| 1/1 [01:44<00:00, 104.41s/it]


Successfully copied all 1 files from cache to `d1-3B-banking77-merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

Splitting model.safetensors (size: 5.82 GB)...


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [02:27<00:00, 36.92s/it]


Unsloth: Regenerating safetensors index...
Unsloth: Merge process complete. Saved to `/content/d1-3B-banking77-merged`


<a name="Inference"></a>
### Evaluation
Let's see how fine-tuning changed d1-3B's decisions. We first free up the GPU memory and measure how well the original d1-3B performs on the 100 test messages. `system_one_batch` packs many messages into each forward pass.

In [22]:
del model, trainer
gc.collect()
torch.cuda.empty_cache()

from tqdm.auto import tqdm

def evaluate(model, dataset, batch_size = 16):
    """Accuracy of the model's intent decisions on `dataset`, and its predictions."""
    predictions = []
    # Evaluate in batches, so the progress bar shows how far we are
    for start in tqdm(range(0, len(dataset), batch_size), desc = "Evaluating"):
        texts = dataset[start:start + batch_size]["text"]
        responses = model.system_one_batch([(text, {"intent": intent_question}) for text in texts])
        predictions += [response["answers"]["intent"]["choice"] for response in responses]
    accuracy = sum(p == y for p, y in zip(predictions, dataset["label_text"])) / len(dataset)
    return accuracy, predictions

base_model = AutoModel.from_pretrained(
    model_id,
    trust_remote_code = True,
    dtype = dtype,
).to("cuda")

baseline_accuracy, baseline_predictions = evaluate(base_model, test_dataset)
print(f"Baseline accuracy: {baseline_accuracy:.1%}")

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/7 [00:00<?, ?it/s]

Baseline accuracy: 83.0%


Now let's free up the GPU memory again and load the fine-tuned model the same way: with `AutoModel` and `trust_remote_code = True`.

In [23]:
del base_model
gc.collect()
torch.cuda.empty_cache()

finetuned = AutoModel.from_pretrained(
    output_dir,
    trust_remote_code = True,
    dtype = dtype,
).to("cuda")

finetuned_accuracy, finetuned_predictions = evaluate(finetuned, test_dataset)
print(f"Baseline accuracy:   {baseline_accuracy:.1%}")
print(f"Fine-tuned accuracy: {finetuned_accuracy:.1%}")

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/7 [00:00<?, ?it/s]

Baseline accuracy:   83.0%
Fine-tuned accuracy: 89.0%
